[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/reinhart-group/generative-copolymer-workshop/blob/main/day1/02_virtual_lab.ipynb)

# Day 1 — Async: The Virtual Lab

**Objectives:**
- Run a Kremer-Grest simulation of your assigned monomer sequence
- Generate trajectory files (`.gsd`) and rendered snapshots
- Apply metadata tags for reproducibility and dataset integration

In [ ]:
#@title  ⚙️  Step 1 of 2 — install conda (causes a restart; that's normal). { display-mode: "form" }
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
#@title  ⚙️  Step 2 of 2 — install the simulation + drawing packages (~2 min). { display-mode: "form" }
%%capture
!conda install -y python=3.12 scipy gsd "hoomd=*=cpu*" freud fresnel pillow
!pip install -q plotly

In [ ]:
#@title  ⚙️  cdse_lab — shared plotting/IO helpers for Day 1. You can ignore this cell. { display-mode: "form" }
#
# This module intentionally contains NO simulation code. HOOMD snapshot setup,
# integrators, thermostats, and forces are written out directly in each notebook
# (01_md_simulation, 02_virtual_lab) -- seeing that code is the lesson. This
# module only hides generic, non-pedagogical plumbing:
#
#     Trajectory                              -- a lightweight "movie": frames + times
#     grab_frame(snapshot)                    -- capture a HOOMD/GSD snapshot into a frame dict
#     unwrap(frame)                           -- undo periodic wrapping (image flags -> continuous coords)
#     save_gsd(traj, filename) / load_gsd(f)  -- write / read a real .gsd file
#     show3d(frame_or_traj) / animate3d(traj) -- interactive 3D picture / movie (Plotly)
#     show_row(*figs)                         -- lay several show3d/animate3d figures side by side
#     measure_size(traj)                      -- radius of gyration vs. time
#     render_pretty(frame_or_traj)            -- optional ray-traced still (needs fresnel)
#
# Forked from reu/cdse_lab.py (a separate, unrelated curriculum) -- this copy is
# scoped to the workshop and does not track that file.

import numpy as np
import gsd.hoomd
import plotly.graph_objects as go

# Colors: type "A" = red (slippery), type "B" = blue (sticky)
_A_COLOR = "crimson"
_B_COLOR = "royalblue"

__all__ = [
    "Trajectory", "grab_frame", "unwrap",
    "save_gsd", "load_gsd",
    "show3d", "show_row", "animate3d", "measure_size", "render_pretty",
]


def _typeids_to_seq(typeids):
    """Inverse of the 'A'/'B' -> 0/1 encoding, for one chain's worth of typeids."""
    return "".join("A" if int(t) == 0 else "B" for t in typeids)


class Trajectory:
    """A lightweight 'movie': a list of frames plus a little bookkeeping.

    Each frame is a dict with:
        position : (N,3) float  -- WRAPPED coordinates (inside the box), as in a real GSD
        image    : (N,3) int    -- how many box-lengths each particle has wandered
        typeid   : (N,)  int     -- 0 = A, 1 = B
        bonds    : (M,2) int     -- pairs of bonded particle indices
        box      : (3,)  float   -- box edge lengths (Lx, Ly, Lz)
    Use unwrap(frame) to get continuous (un-wrapped) coordinates.
    """

    def __init__(self, sequence, num_chains):
        self.sequence = sequence
        self.num_chains = num_chains
        self.chain_len = len(sequence)
        self.frames = []
        self.times = []

    def __len__(self):
        return len(self.frames)

    def __getitem__(self, i):
        return self.frames[i]

    def __repr__(self):
        return (f"Trajectory(sequence={self.sequence!r}, num_chains={self.num_chains}, "
                f"frames={len(self.frames)})")


def grab_frame(snapshot):
    """Copy the bits we need out of a HOOMD/GSD snapshot (wrapped coords + image flags)."""
    return {
        "position": np.array(snapshot.particles.position, dtype=float),
        "image": np.array(snapshot.particles.image, dtype=int),
        "typeid": np.array(snapshot.particles.typeid, dtype=int),
        "bonds": (np.array(snapshot.bonds.group, dtype=int)
                  if snapshot.bonds.N else np.zeros((0, 2), int)),
        "box": np.array(snapshot.configuration.box[:3], dtype=float),
    }


def unwrap(frame):
    """Return continuous (un-wrapped) coordinates: position + image * box.

    HOOMD stores every particle *inside* the box (wrapped) plus an integer
    "image" counting how many box-lengths it has crossed. Undoing the wrap keeps
    bonded chains continuous instead of jumping across the periodic boundary.
    """
    return frame["position"] + frame["image"] * frame["box"]


# --------------------------------------------------------------------------- #
#  GSD files
# --------------------------------------------------------------------------- #
def save_gsd(traj, filename):
    """Write a Trajectory to a real HOOMD GSD file.

    Static topology (types, bonds) is stored once in frame 0; per-particle
    positions and image flags are stored every frame, exactly like HOOMD does.
    """
    with gsd.hoomd.open(name=filename, mode="w") as gsd_file:
        for i, fr in enumerate(traj.frames):
            f = gsd.hoomd.Frame()
            f.configuration.step = int(traj.times[i])
            L = fr["box"]
            f.configuration.box = [L[0], L[1], L[2], 0, 0, 0]
            f.particles.N = int(len(fr["position"]))
            f.particles.position = fr["position"]
            f.particles.image = fr["image"]
            f.particles.typeid = fr["typeid"]
            if i == 0:
                f.particles.types = ["A", "B"]
                nb = int(len(fr["bonds"]))
                f.bonds.N = nb
                f.bonds.types = ["bond"]
                if nb:
                    f.bonds.group = fr["bonds"]
                    f.bonds.typeid = np.zeros(nb, dtype=int)
            gsd_file.append(f)
    return filename


def load_gsd(filename):
    """Read a HOOMD GSD file back into a Trajectory.

    Reconstructs the sequence and chain count from the topology (works for the
    linear chains this course uses: num_chains = N - n_bonds, chain_len = N / num_chains).
    """
    with gsd.hoomd.open(name=filename, mode="r") as gsd_file:
        first = gsd_file[0]
        N = int(first.particles.N)
        bonds0 = (np.array(first.bonds.group, dtype=int)
                  if first.bonds.N else np.zeros((0, 2), int))
        n_bonds = len(bonds0)
        num_chains = max(1, N - n_bonds)
        chain_len = N // num_chains
        typeid0 = np.array(first.particles.typeid, dtype=int)
        sequence = _typeids_to_seq(typeid0[:chain_len])

        traj = Trajectory(sequence, num_chains)
        for frame in gsd_file:
            box = np.array(frame.configuration.box[:3], dtype=float)
            traj.frames.append({
                "position": np.array(frame.particles.position, dtype=float),
                "image": np.array(frame.particles.image, dtype=int),
                "typeid": np.array(frame.particles.typeid, dtype=int),
                "bonds": bonds0,
                "box": box,
            })
            traj.times.append(int(frame.configuration.step))
    return traj


# --------------------------------------------------------------------------- #
#  Visualization
# --------------------------------------------------------------------------- #
def _frame_traces(frame):
    pos = unwrap(frame)
    tid, bonds = frame["typeid"], frame["bonds"]

    bx, by, bz = [], [], []
    for a, b in bonds:
        bx += [pos[a, 0], pos[b, 0], None]
        by += [pos[a, 1], pos[b, 1], None]
        bz += [pos[a, 2], pos[b, 2], None]

    colors = [_A_COLOR if t == 0 else _B_COLOR for t in tid]
    bond_trace = go.Scatter3d(x=bx, y=by, z=bz, mode="lines",
                              line=dict(color="lightgray", width=4),
                              hoverinfo="skip", showlegend=False)
    bead_trace = go.Scatter3d(x=pos[:, 0], y=pos[:, 1], z=pos[:, 2], mode="markers",
                              marker=dict(size=5, color=colors),
                              hoverinfo="skip", showlegend=False)
    return [bond_trace, bead_trace]


def _axis_ranges(frames):
    allpos = np.concatenate([unwrap(f) for f in frames], axis=0)
    lo, hi = allpos.min(axis=0), allpos.max(axis=0)
    pad = 0.1 * (hi - lo + 1.0)
    return [(lo[i] - pad[i], hi[i] + pad[i]) for i in range(3)]


def _scene(ranges):
    xr, yr, zr = ranges
    return dict(xaxis=dict(range=xr, visible=False),
                yaxis=dict(range=yr, visible=False),
                zaxis=dict(range=zr, visible=False),
                aspectmode="data")


def show3d(obj, title=None, return_fig=False):
    """Draw ONE interactive 3D picture. Pass a frame OR a Trajectory (shows last frame).

    return_fig=True returns the Figure instead of showing it, so you can hand
    several to show_row(...) and see them side by side in a single output.
    """
    frame = obj.frames[-1] if isinstance(obj, Trajectory) else obj
    fig = go.Figure(data=_frame_traces(frame))
    fig.update_layout(scene=_scene(_axis_ranges([frame])),
                      margin=dict(l=0, r=0, t=30 if title else 0, b=0),
                      title=title, width=600, height=500)
    if return_fig:
        return fig
    fig.show()


def show_row(*figs, width=440, height=430, gap=8):
    """Show several Plotly figures SIDE BY SIDE in a single output (no vertical stacking).

    Pass figures made with show3d(..., return_fig=True) / animate3d(..., return_fig=True),
    or any go.Figure. Colab caps the output height but allows width, so this lays the
    figures in one horizontal flex row (scrolls sideways if they overflow).
    """
    from IPython.display import HTML, display

    blocks = []
    for k, fig in enumerate(figs):
        fig.update_layout(width=width, height=height)
        blocks.append(fig.to_html(full_html=False,
                                  include_plotlyjs=("cdn" if k == 0 else False)))
    cells = "".join(f'<div style="flex:0 0 auto;">{b}</div>' for b in blocks)
    display(HTML(
        f'<div style="display:flex;flex-wrap:nowrap;gap:{gap}px;'
        f'overflow-x:auto;align-items:flex-start;">{cells}</div>'))


def animate3d(traj, title=None, return_fig=False):
    """Draw a 3D MOVIE with a play button and a slider.

    return_fig=True returns the Figure instead of showing it (see show_row).
    """
    ranges = _axis_ranges(traj.frames)
    fig = go.Figure(
        data=_frame_traces(traj.frames[0]),
        frames=[go.Frame(data=_frame_traces(f), name=str(i))
                for i, f in enumerate(traj.frames)],
    )
    fig.update_layout(
        scene=_scene(ranges),
        margin=dict(l=0, r=0, t=30 if title else 0, b=0),
        title=title, width=600, height=520,
        updatemenus=[dict(type="buttons", showactive=False, x=0.05, y=0.05, xanchor="left",
            buttons=[
                dict(label="▶ Play", method="animate",
                     args=[None, dict(frame=dict(duration=80, redraw=True), fromcurrent=True)]),
                dict(label="⏸ Pause", method="animate",
                     args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")]),
            ])],
        sliders=[dict(active=0, x=0.15, len=0.8, y=0,
            steps=[dict(method="animate", label=str(i),
                        args=[[str(i)], dict(frame=dict(duration=0, redraw=True), mode="immediate")])
                   for i in range(len(traj.frames))])],
    )
    if return_fig:
        return fig
    fig.show()


def measure_size(traj):
    """Return (times, Rg): radius of gyration averaged over chains, per frame."""
    cl, nc = traj.chain_len, traj.num_chains
    times, sizes = [], []
    for frame, t in zip(traj.frames, traj.times):
        pos = unwrap(frame).reshape(nc, cl, 3)
        com = pos.mean(axis=1, keepdims=True)
        rg2 = ((pos - com) ** 2).sum(axis=2).mean(axis=1)
        sizes.append(np.sqrt(rg2).mean())
        times.append(t)
    return np.array(times), np.array(sizes)


def render_pretty(obj):
    """Optional: a fancy ray-traced still image (needs fresnel). Pass a frame or trajectory."""
    import fresnel, PIL.Image

    frame = obj.frames[-1] if isinstance(obj, Trajectory) else obj
    pos = unwrap(frame)
    tid, bonds = frame["typeid"], frame["bonds"]
    N = len(pos)

    scene = fresnel.Scene()
    colors = np.empty((N, 3))
    colors[tid == 0] = fresnel.color.linear([0.95, 0, 0])
    colors[tid == 1] = fresnel.color.linear([0, 0, 0.95])
    geo = fresnel.geometry.Sphere(scene, N=N, radius=0.3)
    geo.position[:] = pos
    geo.material = fresnel.material.Material(roughness=0.9)
    geo.outline_width = 0.05
    geo.material.primitive_color_mix = 1.0
    geo.color[:] = fresnel.color.linear(colors)

    if len(bonds):
        ends = np.stack([pos[bonds[:, 0]], pos[bonds[:, 1]]], axis=1)
        cyl = fresnel.geometry.Cylinder(scene, N=len(bonds))
        cyl.material = fresnel.material.Material(
            roughness=0.5, color=fresnel.color.linear([0.8, 0.8, 0.8]))
        cyl.points[:] = ends
        cyl.radius[:] = [0.12] * len(bonds)

    scene.background_color = (1, 1, 1)
    scene.background_alpha = 1
    scene.camera = fresnel.camera.Orthographic.fit(scene, view="isometric", margin=0.1)
    out = fresnel.preview(scene, w=600, h=600)
    return PIL.Image.fromarray(out[:, :, 0:3], mode="RGB")

In [ ]:
import hoomd
import gsd, gsd.hoomd
import numpy as np
import freud
import fresnel, PIL

## 1. Your Assigned Sequence

TODO: Sequence assignment and encoding

In [ ]:
# TODO: Define your assigned sequence
sequence = "0111011110"  # Replace with your assignment

## 2. Running the Simulation

TODO: Scaffolded simulation setup and execution
frame = hoomd.Snapshot() # Code that will setup simulation initial conditions and parameters
num_chains = 200
frame.particles.N = 10*num_chains
frame.particles.types = ["A","B"]
frame.particles.mass[:] =  1.0
frame.bonds.N = 9*num_chains
frame.bonds.types = ['bond']
types = []
pos= []
pairs = []
half = -1
it_x = 0
it_y = 0
Create initial configuration for polymer chains
for chain in range(num_chains):
  init_ind = chain*10
  for mon in range(10):
    mon_ind = init_ind + mon
    pos.append([it_x,it_y,(half*(mon+(half+1)/2))])
    if len(sequence) != 10:
      raise ValueError('The input sequence is not the correct length. Check that your sequence hasa length of 10')
    types.append(int(sequence[mon]))
    if mon < (9):
      pairs.append([mon_ind,mon_ind+1])
  if half == 1:
    if it_x < 0 and it_x > -10:
      it_x = -it_x
    elif it_x >= 0 and it_x < 9:
      it_x = -(it_x+1)
    else:
      it_x = 0
      if it_y < 0:
        it_y = -it_y
      else:
        it_y = -(it_y+1)

  half *=-1
frame.bonds.group[:] = pairs
frame.particles.position[:] = pos
frame.particles.typeid[:] = types
frame.configuration.box = [23.5,23.5,23.5,0,0,0]


set up simulation
device = hoomd.device.CPU()
sim = hoomd.Simulation(device=device,seed=42)
sim.create_state_from_snapshot(frame)

pair and bonded interactions: A=purely WCA, B=LJ
nl = hoomd.md.nlist.Cell(buffer=0.4)
lj = hoomd.md.pair.LJ(nlist=nl, default_r_cut=2.5)
lj.params[('A','A')] = dict(epsilon = 1.0, sigma= 1.0)
lj.params[('A','B')] = dict(epsilon = 1.0, sigma= 1.0)
lj.params[('B','B')] = dict(epsilon = 1.0, sigma= 1.0)
lj.r_cut[('A','A')] = 2**(1/6)
lj.r_cut[('A','B')] = 2**(1/6)
lj.mode='shift'

fenewca = hoomd.md.bond.FENEWCA()
fenewca.params['bond'] = dict(k=30.0, r0=1.5, epsilon=1.0, sigma=1.0, delta=0.0)

Set up thermostat and integrator
sim.state.thermalize_particle_momenta(filter=hoomd.filter.All(), kT=1.0)
nvt = hoomd.md.methods.Langevin(filter=hoomd.filter.All(), kT=1.0)
integrator = hoomd.md.Integrator(dt=0.005, methods=[nvt], forces = [lj,fenewca])
sim.operations.integrator = integrator

thermo = hoomd.md.compute.ThermodynamicQuantities(filter=hoomd.filter.All())
sim.operations += thermo
logger = hoomd.logging.Logger(categories=['scalar'])
logger.add(sim, quantities=['timestep'])
logger.add(thermo, quantities = ['kinetic_energy', 'kinetic_temperature', 'potential_energy'])

table = hoomd.write.Table(trigger=hoomd.trigger.Periodic(12500),
    logger=logger, pretty=False,max_precision=15,max_header_len=30)

sim.operations.writers.append(table)

In [ ]:
n_frames = 24
chunk = max(1, 125000 // n_frames)
traj = Trajectory(sequence=sequence, num_chains=num_chains)
traj.frames.append(grab_frame(sim.state.get_snapshot()))
traj.times.append(sim.timestep)
for _ in range(n_frames):
    sim.run(chunk)
    traj.frames.append(grab_frame(sim.state.get_snapshot()))
    traj.times.append(sim.timestep)

# Save final system structure to .gsd file
filename = 'test.gsd' # set filename for .gsd output file
hoomd.write.GSD.write(sim.state,filename=filename,filter=hoomd.filter.All(),mode='wb')

## 3. Visualization and Data Organization

TODO: tag metadata

In [ ]:
animate3d(traj)